<a href="https://colab.research.google.com/github/DhimanTarafdar/AAA/blob/main/Cat_vs_Dog_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import kagglehub
import matplotlib.pyplot as plt

# **Download Dataset**

In [2]:
path = kagglehub.dataset_download("salader/dogsvscats")
print("path", path)

Using Colab cache for faster access to the 'dogsvscats' dataset.
path /kaggle/input/dogsvscats


# **Setup**

In [3]:
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


# **Path Setup**

In [4]:
TRAIN_PATH = os.path.join(path, "train")
VAL_PATH   = os.path.join(path, "test")

print("Train Path:", TRAIN_PATH)
print("Val Path  :", VAL_PATH)

Train Path: /kaggle/input/dogsvscats/train
Val Path  : /kaggle/input/dogsvscats/test


# **Image Transformation**

In [5]:
transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

# Image Transformation Pipeline

`transforms.Compose()` একটা pipeline বানায় — image একটার পর একটা step এর মধ্য দিয়ে যায়।

## ৩টা step:

- **Resize((128,128))** → যেকোনো size এর image কে 128×128 pixel এ convert করে, কারণ CNN এ সব input এর size same হতে হয়

- **ToTensor()** → PIL image (0-255) কে PyTorch tensor এ convert করে এবং pixel value কে 0.0-1.0 range এ নিয়ে আসে

- **Normalize(mean=[0.5]*3, std=[0.5]*3)** → pixel value কে 0.0-1.0 থেকে -1.0 থেকে +1.0 range এ নিয়ে আসে
  - formula: `(pixel - 0.5) / 0.5`
  - `*3` মানে RGB তিনটা channel এর জন্যই একই value

**কেন Normalize করি?** → model training এ gradient stable থাকে, faster converge করে

# **Custom Dataset**

In [9]:
class BinaryClassification(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()

        self.samples = []
        self.transform = transform

        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])

        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}

        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                if os.path.isfile(img_path):
                    label = self.class_to_idx[class_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# Custom Dataset Class — BinaryClassification

PyTorch এ নিজের data load করতে হলে `Dataset` class কে inherit করে নিজের class বানাতে হয়।

> Plant disease এ class এর নাম ছিল `MultiClassClassfication` কারণ সেখানে 38টা class ছিল।
> Cat vs Dog এ মাত্র 2টা class (cat, dog) তাই নাম দিলাম `BinaryClassification`।

---

## `__init__()` — class তৈরির সময় যা যা হয়

**Step 1 — খালি list এবং transform রেখে দাও**
- `self.samples = []` → এখানে (image_path, label) pair জমা হবে
- `self.transform = transform` → image এ কী কী করতে হবে সেটা রেখে দিলাম

**Step 2 — সব class এর নাম বের করো**
- `root_dir` এর ভেতরে যত folder আছে সেগুলোই class — এখানে `cat` এবং `dog`
- `sorted()` দিয়ে alphabetically সাজানো হলো — তাই `cat → 0`, `dog → 1`

**Step 3 — প্রতিটা class কে একটা number দাও**
- CNN সরাসরি string বোঝে না, তাই class এর নামকে unique number এ convert করা হলো
- result: `{'cat': 0, 'dog': 1}`

**Step 4 — সব image এর path আর label একসাথে জমা করো**
- প্রতিটা class folder এ যাও → সেই folder এর প্রতিটা file দেখো → নিশ্চিত হও এটা file না folder → class নাম কে number এ convert করো → list এ জমা করো
- শেষে `self.samples` এ থাকে: `[('root/cat/img1.jpg', 0), ('root/dog/img4.jpg', 1), ...]`

---

## `__len__()` — dataset এ মোট কতটা image আছে
- DataLoader এই method কে call করে জানে যে loop কতবার চালাতে হবে

---

## `__getitem__()` — index দিলে একটা image আর তার label দাও
- `self.samples[idx]` থেকে path আর label নাও
- disk থেকে image পড়ো, `.convert("RGB")` দিয়ে নিশ্চিত করো সব image 3 channel এ আছে
- transform apply করো (resize, tensor, normalize)
- DataLoader যখন batch বানায়, সে বারবার এই method কে আলাদা আলাদা index দিয়ে call করে

---

## সহজ ভাষায় পুরো flow

```
root_dir/
├── cat/        ← class 0
│   ├── img1.jpg
│   └── img2.jpg
└── dog/        ← class 1
    └── img3.jpg

__init__    → সব image এর path + label একটা list এ জমা করো
__len__     → বলো list এ কতটা আছে
__getitem__ → index দিলে সেই image টা পড়ো, transform করো, return করো
```

# **Load Data**

In [10]:
train_dataset_full = BinaryClassification(TRAIN_PATH, transform)
test_dataset_full  = BinaryClassification(VAL_PATH, transform)
num_classes        = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Class mapping    :", train_dataset_full.class_to_idx)
print("Full train size  :", len(train_dataset_full))
print("Full test size   :", len(test_dataset_full))

Number of classes: 2
Class mapping    : {'cats': 0, 'dogs': 1}
Full train size  : 20000
Full test size   : 5000


# **Subset + DataLoader**

In [18]:
train_dataset = Subset(train_dataset_full, list(range(min(20000, len(train_dataset_full)))))
test_dataset  = Subset(test_dataset_full,  list(range(min(5000, len(test_dataset_full)))))

print("Train subset size:", len(train_dataset))
print("Test subset size :", len(test_dataset))

Train subset size: 20000
Test subset size : 5000


In [12]:
pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  pin_memory=pin)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, pin_memory=pin)

# **CNN Architecture**

In [19]:
class MyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*16*16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = MyCNN(num_classes=num_classes).to(device)
print(model)

MyCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (9): ReLU()
    (10): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, 

# CNN Architecture — MyCNN

এই model দুইটা ভাগে কাজ করে —
**features** → image থেকে pattern বের করো
**classifier** → সেই pattern দেখে সিদ্ধান্ত নাও

---

## Part 1 — Feature Extractor (3টা Conv Block)

প্রতিটা block এ ৪টা step: `Conv2d → ReLU → BatchNorm → MaxPool`

**Conv2d — pattern খোঁজো**
- `Conv2d(3, 32, kernel_size=3, padding='same')` → 3টা channel (RGB) নিয়ে 32টা আলাদা filter দিয়ে scan করে, প্রতিটা filter একটা আলাদা pattern খোঁজে (যেমন edge, texture, color spot)
- `padding='same'` → output এর size input এর মতোই থাকে
- block 1: 3 → 32, block 2: 32 → 64, block 3: 64 → 128 — প্রতিটা block এ filter বাড়ে কারণ deeper layer এ complex pattern ধরতে হয়

**ReLU — negative value বাদ দাও**
- সব negative value কে 0 বানিয়ে দেয়
- এটা না থাকলে পুরো network টা একটা simple linear equation হয়ে যেত, complex pattern শিখতে পারত না

**BatchNorm2d — value গুলো normalize করো**
- প্রতিটা batch এর output কে normalize করে রাখে যাতে training stable থাকে এবং দ্রুত শেখে

**MaxPool2d(2) — image ছোট করো**
- প্রতি 2×2 block থেকে সবচেয়ে বড় value টা রাখে, বাকি বাদ
- image এর size অর্ধেক হয়ে যায়: 128×128 → 64×64 → 32×32 → 16×16

---

## Part 2 — Classifier (Decision Maker)

feature extractor এর পর image এর shape হয় `128 × 16 × 16`

**Flatten — 3D কে 1D বানাও**
- `128 × 16 × 16 = 32768` → একটা লম্বা list এ পরিণত হয়
- কারণ Linear layer শুধু 1D input নেয়

**Linear — pattern দেখে সিদ্ধান্ত নাও**
- `Linear(32768, 256)` → 32768টা value থেকে 256টা important feature বের করো
- `Linear(256, 128)` → আরো compress করো
- `Linear(128, num_classes)` → শেষে প্রতিটা class এর জন্য একটা score দাও

> Plant disease এ Linear(32768, 128) ছিল — Cat vs Dog এ Linear(32768, 256) করা হয়েছে কারণ বেশি data নিচ্ছি তাই model এর capacity একটু বাড়ানো দরকার

**Dropout(0.5)** — training এর সময় randomly 50% neuron বন্ধ রাখে
- Plant disease এ Dropout(0.4) ছিল — এখানে 0.5 করা হয়েছে কারণ overfitting আরো ভালোভাবে রোধ করতে চাই

---

## পুরো flow এক নজরে

```
Input Image (3 × 128 × 128)
        ↓
Conv Block 1 → (32 × 128 × 128) → MaxPool → (32 × 64 × 64)
        ↓
Conv Block 2 → (64 × 64 × 64)  → MaxPool → (64 × 32 × 32)
        ↓
Conv Block 3 → (128 × 32 × 32) → MaxPool → (128 × 16 × 16)
        ↓
Flatten → (32768,)
        ↓
Linear → 256 → Dropout(0.5)
        ↓
Linear → 128 → Dropout(0.5)
        ↓
Linear → 2  ← cat score, dog score
```

# **Model Summary**

In [20]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters    : {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters    : 8,515,714
Trainable Parameters: 8,515,714


# **Training Setup**

In [21]:
learning_rate = 0.001
epochs        = 10
criterion     = nn.CrossEntropyLoss()
optimizer     = optim.Adam(model.parameters(), lr=learning_rate)

# **Training Loop**

In [22]:
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels   = batch_labels.to(device)

        outputs = model(batch_features)
        loss    = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 0.0063
Epoch 2/10, Loss: 0.0000
Epoch 3/10, Loss: 0.0000
Epoch 4/10, Loss: 0.0000
Epoch 5/10, Loss: 0.0000
Epoch 6/10, Loss: 0.0000
Epoch 7/10, Loss: 0.0000
Epoch 8/10, Loss: 0.0000
Epoch 9/10, Loss: 0.0000
Epoch 10/10, Loss: 0.0000
